In [5]:
import json, os, sys, re, copy
sys.path.append(os.path.abspath(".."))
from simulator.game_ui import run_replay_viewer

In [6]:
persistent_state = {
  "turn": 1,

  "me_life": 8000,
  "me_cards_in_deck": 40,
  "me_cards_in_extra_deck": 15,

  "me_hand": [],
  "me_GY": [],
  "me_banished": [],
  "me_banish": [],
  "me_field_spell": None,

  "me_field": {
    "M-1": None,
    "M-2": None,
    "M-3": None,
    "M-4": None,
    "M-5": None,

    "S-1": None,
    "S-2": None,
    "S-3": None,
    "S-4": None,
    "S-5": None,
  },

  "me_phase": None,

  "op_life": 8000,
  "op_cards_in_deck": 40,
  "op_cards_in_extra_deck": 15,

  "op_hand": [],
  "op_GY": [],
  "op_banished": [],
  "op_banish": [],

  "op_field_spell": None,

  "op_field": {
    "M-1": None,
    "M-2": None,
    "M-3": None,
    "M-4": None,
    "M-5": None,

    "S-1": None,
    "S-2": None,
    "S-3": None,
    "S-4": None,
    "S-5": None,
  },

  "op_phase": None
}

# make a structure for a card with all fields set to None or default values
default_card = {
  "name": None,
  "object_id": None,
  "battle_position": None,
  "face_up": None,
  "owner": None
}


example_card = {
  "name": "Unknown Card",
  "object_id": 0,
  "battle_position": None,
  "face_up": True,
  "owner": "p1"
}

In [ ]:
game_log = json.load(open("../games/replay.json", "r", encoding="utf-8"))

player_1 = game_log["player1"]["username"]
p1_main_total = game_log["player1"]["main_total"]
p1_extra_total = game_log["player1"]["extra_total"]
player_2 = game_log["player2"]["username"]
p2_main_total = game_log["player2"]["main_total"]
p2_extra_total = game_log["player2"]["extra_total"]

persistent_state["me_cards_in_deck"] = p1_main_total
persistent_state["me_cards_in_extra_deck"] = p1_extra_total
persistent_state["op_cards_in_deck"] = p2_main_total
persistent_state["op_cards_in_extra_deck"] = p2_extra_total


STATE_ACTION_PAIRS = []

for play in game_log["plays"]:
    play_type = play["play"]

    # Skipping
    if play_type in ["Good", "Shuffle hand", "RPS", "Attack directly", "View deck", "Stop viewing", "Shuffle deck", "Declare", "View GY", "View GY 2", "Duel message"]:
        print(f"⏭️ Skipping play: {play}")
    
    # Real actions
    elif play_type == "Pick first":
        print(f"✅ Processing play: {play}")
        cards = play["cards"]
        card_names = [
            {"name": card["name"], "object_id": card["object_id"], "battle_position": None, "face_up": None} for card in cards]
        # add owner p1 to the first 5 cards, owner p2 to the next 5 cards
        card_names[:5] = [{**card, "owner": "p1"} for card in card_names[:5]]
        card_names[5:10] = [{**card, "owner": "p2"} for card in card_names[5:10]]
        print(f"Initial hand for {player_1}: {[card for card in card_names[:5]]}")
        print(f"Initial hand for {player_2}: {[card for card in card_names[5:10]]}")
        persistent_state["me_hand"] = card_names[:5]
        persistent_state["op_hand"] = card_names[5:10]
        persistent_state["me_cards_in_deck"] -= 5
        persistent_state["op_cards_in_deck"] -= 5
    elif play_type == "Draw card":
        print(f"✅ Processing play: {play}")
        card = play["card"]
        drawn_card = {"name": card["name"], "object_id": card["object_id"], "battle_position": None, "face_up": None}
        if play["username"] == player_1:
            drawn_card["owner"] = "p1"
            persistent_state["me_hand"].append(drawn_card)
            persistent_state["me_cards_in_deck"] -= 1
        else:
            drawn_card["owner"] = "p2"
            persistent_state["op_hand"].append(drawn_card)
            persistent_state["op_cards_in_deck"] -= 1
    elif play_type == "Enter DP":
        print(f"✅ Processing play: {play}")
        if play["username"] == player_1:
            persistent_state["me_phase"] = "DP"
        else:
            persistent_state["op_phase"] = "DP"
    elif play_type == "Enter SP":
        print(f"✅ Processing play: {play}")
        if play["username"] == player_1:
            persistent_state["me_phase"] = "SP"
        else:
            persistent_state["op_phase"] = "SP"
    elif play_type == "Enter M1":
        print(f"✅ Processing play: {play}")
        if play["username"] == player_1:
            persistent_state["me_phase"] = "M1"
        else:
            persistent_state["op_phase"] = "M1"
    elif play_type == "Enter BP":
        print(f"✅ Processing play: {play}")
        if play["username"] == player_1:
            persistent_state["me_phase"] = "BP"
        else:
            persistent_state["op_phase"] = "BP"
    elif play_type == "Enter M2":
        print(f"✅ Processing play: {play}")
        if play["username"] == player_1:
            persistent_state["me_phase"] = "M2"
        else:
            persistent_state["op_phase"] = "M2"
    elif play_type == "Enter EP":
        print(f"✅ Processing play: {play}")
        if play["username"] == player_1:
            persistent_state["me_phase"] = "EP"
        else:
            persistent_state["op_phase"] = "EP"
    elif play_type == "End turn":
        print(f"✅ Processing play: {play}")
        if play["username"] == player_1:
            persistent_state["me_phase"] = "Opponent's turn"
            persistent_state["op_phase"] = "DP"
        else:
            persistent_state["op_phase"] = "Opponent's turn"
            persistent_state["me_phase"] = "DP"
        persistent_state["turn"] += 1
    elif play_type == "Start turn":
        print(f"✅ Processing play: {play}")
        if play["username"] == player_1:
            persistent_state["me_phase"] = "DP"
            persistent_state["op_phase"] = "Opponent's turn"
        else:
            persistent_state["op_phase"] = "DP"
            persistent_state["me_phase"] = "Opponent's turn"

    elif play_type == "Life points":
        print(f"✅ Processing play: {play}")
        if play["username"] == player_1:
            persistent_state["me_life"] +=play["amount"]
        else:
            persistent_state["op_life"] += play["amount"]
    elif play_type == "Set ST":
        print(f"✅ Processing play: {play}")
        log = play["log"]["private_log"] if "private_log" in play["log"] else play["log"]["public_log"]
        pattern = r'Set "(.*?)" from (\w+) .* to (\S+)'
        match = re.search(pattern, log)

        if match:
            card_name = match.group(1)
            start = match.group(2)
            finish = match.group(3)

        card_object = None
        if start == "hand":
            if play["username"] == player_1:
                # find the card in hand and remove it
                for card in persistent_state["me_hand"]:
                    if card["name"] == card_name:
                        card_object = card
                        persistent_state["me_hand"].remove(card)
                        break
                # place it on the field
                persistent_state["me_field"][finish] = {"name": card_name, "object_id": card_object["object_id"] if card_object else None, "battle_position": None, "face_up": False, "owner": card_object["owner"] if card_object else None}
            else:
                # find the card in hand and remove it
                for card in persistent_state["op_hand"]:
                    if card["name"] == card_name:
                        card_object = card
                        persistent_state["op_hand"].remove(card)
                        break
                # place it on the field
                persistent_state["op_field"][finish] = {"name": card_name, "object_id": card_object["object_id"] if card_object else None, "battle_position": None, "face_up": False, "owner": card_object["owner"] if card_object else None}
    
    elif play_type == "Activate ST":
        print(f"✅ Processing play: {play}")
        log = (play["log"].get("private_log") or play["log"].get("public_log") or "").strip()

        # Handles:
        # Activated "Heavy Storm" from hand (2/5) to S-3
        # Activated Set "Spiritual Water Art - Aoi" in S-4
        pattern = r'^Activated(?: Set)?\s+"([^"]+)"(?:\s+from\s+(\w+)(?:\s+\(\d+/\d+\))?\s+to\s+(S-\d+)|\s+in\s+(S-\d+))\s*$'
        m = re.search(pattern, log)

        if not m:
            print(f"⚠️ Could not parse Activate ST log: {log!r}")
        else:
            card_name = m.group(1)
            start = (m.group(2) or "").lower() or None
            finish = m.group(3) or m.group(4)   # S-#

            card_object = None

            # Activate from hand -> remove from hand
            if start == "hand":
                if play["username"] == player_1:
                    for card in persistent_state["me_hand"]:
                        if card["name"] == card_name:
                            card_object = card
                            persistent_state["me_hand"].remove(card)
                            break
                else:
                    for card in persistent_state["op_hand"]:
                        if card["name"] == card_name:
                            card_object = card
                            persistent_state["op_hand"].remove(card)
                            break

            # If it was "Activated Set ... in S-x", the card is already on field facedown.
            # In both cases, mark zone as face-up (activated).
            if play["username"] == player_1:
                persistent_state["me_field"][finish] = {
                    "name": card_name,
                    "object_id": card_object["object_id"] if card_object else persistent_state["me_field"].get(finish, {}).get("object_id"),
                    "battle_position": None,
                    "face_up": True,
                    "owner": card_object["owner"] if card_object else None
                }
            else:
                persistent_state["op_field"][finish] = {
                    "name": card_name,
                    "object_id": card_object["object_id"] if card_object else persistent_state["op_field"].get(finish, {}).get("object_id"),
                    "battle_position": None,
                    "face_up": True,
                    "owner": card_object["owner"] if card_object else None
                }
        
    
    elif play_type == "Set Field Spell":
        print(f"✅ Processing play: {play}")
        log = (play["log"].get("private_log") or play["log"].get("public_log") or "").strip()

        # Handles BOTH:
        # Set Field Spell "A Legendary Ocean" from hand (7/7)
        # Set Field Spell "A Legendary Ocean" in Field Spell Zone
        pattern = r'^Set Field Spell\s+"([^"]+)"(?:\s+from\s+([A-Za-z]+)\s+\(\d+/\d+\)|\s+in Field Spell Zone)\s*$'
        match = re.search(pattern, log)

        if not match:
            print(f"⚠️ Could not parse Set Field Spell log: {log!r}")
        else:
            card_name = match.group(1)
            start = match.group(2).lower() if match.group(2) else None
            finish = "FZ"  # your field-spell zone key; change if you use something else

            card_object = None

            if play["username"] == player_1:
                if start == "hand":
                    for card in persistent_state["me_hand"]:
                        if card["name"] == card_name:
                            card_object = card
                            persistent_state["me_hand"].remove(card)
                            break

                # place into my field spell zone (set = face_down)
                persistent_state["me_field_spell"] = {
                    "name": card_name,
                    "object_id": card_object["object_id"] if card_object else persistent_state.get("me_field_spell", {}).get("object_id"),
                    "battle_position": None,
                    "face_up": False,
                    "owner": card_object["owner"] if card_object else None
                }

            else:
                if start == "hand":
                    for card in persistent_state["op_hand"]:
                        if card["name"] == card_name:
                            card_object = card
                            persistent_state["op_hand"].remove(card)
                            break

                # place into opponent field spell zone (set = face_down)
                persistent_state["op_field_spell"] = {
                    "name": card_name,
                    "object_id": card_object["object_id"] if card_object else persistent_state.get("op_field_spell", {}).get("object_id"),
                    "battle_position": None,
                    "face_up": False,
                    "owner": card_object["owner"] if card_object else None
                }
    elif play_type == "Activate Field Spell":
        print(f"✅ Processing play: {play}")
        log = (play["log"].get("private_log") or play["log"].get("public_log") or "").strip()

        # Handles ALL formats:
        # Activated Field Spell "X" from Deck
        # Activated Field Spell "X" from hand (3/3)
        # Activated Field Spell Set "X" in Field Spell Zone
        pattern = r'^Activated Field Spell(?: Set)?\s+"([^"]+)"(?:\s+from\s+([A-Za-z]+)(?:\s+\(\d+/\d+\))?|\s+in Field Spell Zone)\s*$'
        match = re.search(pattern, log)

        if not match:
            print(f"⚠️ Could not parse Activate Field Spell log: {log!r}")
        else:
            card_name = match.group(1)
            start = match.group(2).lower() if match.group(2) else None

            card_object = None

            if play["username"] == player_1:

                if start == "hand":
                    for card in persistent_state["me_hand"]:
                        if card["name"] == card_name:
                            card_object = card
                            persistent_state["me_hand"].remove(card)
                            break

                elif start == "deck":
                    persistent_state["me_cards_in_deck"] -= 1

                persistent_state["me_field_spell"] = {
                    "name": card_name,
                    "object_id": card_object["object_id"] if card_object else None,
                    "battle_position": None,
                    "face_up": True,
                    "owner": "p1"
                }

            else:

                if start == "hand":
                    for card in persistent_state["op_hand"]:
                        if card["name"] == card_name:
                            card_object = card
                            persistent_state["op_hand"].remove(card)
                            break

                elif start == "deck":
                    persistent_state["op_cards_in_deck"] -= 1

                persistent_state["op_field_spell"] = {
                    "name": card_name,
                    "object_id": card_object["object_id"] if card_object else None,
                    "battle_position": None,
                    "face_up": True,
                    "owner": "p2"
                }

    elif play_type == "Normal Summon":
        print(f"✅ Processing play: {play}")
        log = play["log"].get("private_log") or play["log"].get("public_log") or ""

        pattern = r'Normal Summoned\s+"(.*?)"\s+from\s+(\w+)\s+\(\d+/\d+\)\s+to\s+(\S+)'
        match = re.search(pattern, log)

        if not match:
            print(f"⚠️ Could not parse Normal Summon log: {log!r}")
        else:
            card_name = match.group(1)
            start = match.group(2)
            finish = match.group(3)

            card_object = None
            if start == "hand":
                if play["username"] == player_1:
                    for card in persistent_state["me_hand"]:
                        if card["name"] == card_name:
                            card_object = card
                            persistent_state["me_hand"].remove(card)
                            break
                    persistent_state["me_field"][finish] = {
                        "name": card_name,
                        "object_id": card_object["object_id"] if card_object else None,
                        "battle_position": "ATK",
                        "face_up": True,
                        "owner": card_object["owner"] if card_object else None
                    }
                else:
                    for card in persistent_state["op_hand"]:
                        if card["name"] == card_name:
                            card_object = card
                            persistent_state["op_hand"].remove(card)
                            break
                    persistent_state["op_field"][finish] = {
                        "name": card_name,
                        "object_id": card_object["object_id"] if card_object else None,
                        "battle_position": "ATK",
                        "face_up": True,
                        "owner": card_object["owner"] if card_object else None
                    }

    elif play_type == "To GY":
        print(f"✅ Processing play: {play}")

        log = (play["log"].get("private_log") or play["log"].get("public_log") or "").strip()

        # Matches ALL formats:
        # Sent "Machina Fortress" from hand (1/4) to GY
        # Sent "Moray of Greed" from S-4 to GY
        # Sent Set "Brain Control" from S-3 to GY
        # Sent "A Legendary Ocean" from Field Spell Zone to GY
        # Sent "Terraforming" from Deck to GY
        pattern = r'^Sent(?: Set)?\s+"([^"]+)"\s+from\s+(.+?)\s+to\s+GY$'
        match = re.search(pattern, log)

        if not match:
            print(f"⚠️ Could not parse To GY log: {log!r}")
            continue

        card_name = match.group(1)
        source_raw = match.group(2).strip()

        # -------- normalize source --------
        if source_raw.startswith("hand"):
            source = "hand"
        elif source_raw.startswith("Deck"):
            source = "Deck"
        elif source_raw.startswith("Field Spell Zone"):
            source = "Field Spell Zone"
        else:
            source = source_raw  # S-3, M-2, etc.

        is_me = play["username"] == player_1

        # -------- select correct zones --------
        if is_me:
            hand = persistent_state["me_hand"]
            field = persistent_state["me_field"]
            gy = persistent_state["me_GY"]
            field_spell = persistent_state.get("me_field_spell")
        else:
            hand = persistent_state["op_hand"]
            field = persistent_state["op_field"]
            gy = persistent_state["op_GY"]
            field_spell = persistent_state.get("op_field_spell")

        card_object = None

        # -------- remove from hand --------
        if source == "hand":
            for card in hand:
                if card["name"] == card_name:
                    card_object = card
                    hand.remove(card)
                    break

        # -------- remove from field --------
        elif source in field:
            card_object = field.pop(source)

        # -------- remove from field spell zone --------
        elif source == "Field Spell Zone":
            card_object = field_spell
            if is_me:
                persistent_state["me_field_spell"] = None
            else:
                persistent_state["op_field_spell"] = None

        # -------- sent from deck --------
        elif source == "Deck":
            card_object = {
                "name": card_name,
                "object_id": play.get("card", {}).get("object_id"),
                "battle_position": None,
                "face_up": True,
                "owner": "p1"if is_me else "p2"
            }

            if is_me:
                persistent_state["me_cards_in_deck"] = max(
                    0, persistent_state["me_cards_in_deck"] - 1
                )
            else:
                persistent_state["op_cards_in_deck"] = max(
                    0, persistent_state["op_cards_in_deck"] - 1
                )

        # -------- already in GY safety --------
        elif source == "GY":
            for card in gy:
                if card["name"] == card_name:
                    card_object = card
                    break

        # -------- fallback (never fail) --------
        if card_object is None:
            card_object = {
                "name": card_name,
                "object_id": play.get("card", {}).get("object_id"),
                "battle_position": None,
                "face_up": True,
                "owner": "p1" if is_me else "p2"
            }

        # -------- add to graveyard --------
        gy = persistent_state["me_GY"] if card_object["owner"] == "p1" else persistent_state["op_GY"]
        gy.append(card_object)

    elif play_type == "To hand":
        print(f"✅ Processing play: {play}")

        log = (play["log"].get("private_log") or play["log"].get("public_log") or "").strip()

        # Matches ALL formats:
        # Added "A Legendary Ocean" from Deck to hand
        # Returned "Heavy Storm" from S-3 to hand
        # Returned "Moray of Greed" from GY to hand
        # Returned "A Legendary Ocean" from Field Spell Zone to hand
        pattern = r'^(Added|Returned)\s+"([^"]+)"\s+from\s+(.+?)\s+to\s+hand$'
        match = re.search(pattern, log)

        if not match:
            print(f"⚠️ Could not parse To hand log: {log!r}")
        else:
            action_type = match.group(1)   # Added or Returned
            card_name = match.group(2)
            source = match.group(3)

            is_me = play["username"] == player_1

            if is_me:
                hand = persistent_state["me_hand"]
                field = persistent_state["me_field"]
                gy = persistent_state["me_GY"]
                field_spell = persistent_state.get("me_field_spell")
            else:
                hand = persistent_state["op_hand"]
                field = persistent_state["op_field"]
                gy = persistent_state["op_GY"]
                field_spell = persistent_state.get("op_field_spell")

            card_object = None

            # ---------- FROM FIELD ZONE ----------
            if source in field:
                card_object = field.pop(source)

            # ---------- FROM FIELD SPELL ZONE ----------
            elif source == "Field Spell Zone":
                card_object = field_spell
                if is_me:
                    persistent_state["me_field_spell"] = None
                else:
                    persistent_state["op_field_spell"] = None

            # ---------- FROM GRAVEYARD ----------
            elif source == "GY":
                for card in gy:
                    if card["name"] == card_name:
                        card_object = card
                        gy.remove(card)
                        break

            # ---------- FROM DECK ----------
            elif source == "Deck":
                card_object = {
                    "name": card_name,
                    "object_id": play.get("card", {}).get("object_id"),
                    "battle_position": None,
                    "face_up": True,
                    "owner": "p1" if is_me else "p2"
                }

                if is_me:
                    persistent_state["me_cards_in_deck"] -= 1
                else:
                    persistent_state["op_cards_in_deck"] -= 1

            # ---------- FALLBACK ----------
            if card_object is None:
                card_object = {
                    "name": card_name,
                    "object_id": play.get("card", {}).get("object_id"),
                    "battle_position": None,
                    "face_up": True,
                    "owner": "p1" if is_me else "p2"
                }

            # ---------- ADD TO HAND ----------
            hand = persistent_state["me_hand"] if card_object["owner"] == "p1" else persistent_state["op_hand"]
            hand.append(card_object)

    elif play_type == "Move":
        print(f"✅ Processing play: {play}")

        log = (play["log"].get("private_log") or play["log"].get("public_log") or "").strip()
        print(f"Parsing Move log: {log!r}")

        pattern = r'^Moved\s+"([^"]+)"\s+from\s+(\S+)\s+to\s+(\S+)$'
        m = re.search(pattern, log)

        if not m:
            print(f"❌ Could not parse Move log: {log!r}")
        else:
            card_name  = m.group(1)
            source_raw = m.group(2)
            dest_raw   = m.group(3)

            # mover side (global persistent_state me/op)
            mover_is_me = (play.get("username") == player_1)
            mover_side = "me" if mover_is_me else "op"
            other_side = "op" if mover_side == "me" else "me"

            def side_from_zone_relative_to_mover(zone: str) -> str:
                # M2-/S2- means "other side relative to mover"
                if zone.startswith(("M2-", "S2-")):
                    return other_side
                return mover_side

            def normalize_zone(zone: str) -> str:
                # M2-3 -> M-3, S2-4 -> S-4
                return zone.replace("M2-", "M-").replace("S2-", "S-")

            source_side = side_from_zone_relative_to_mover(source_raw)
            dest_side   = side_from_zone_relative_to_mover(dest_raw)

            source_zone = normalize_zone(source_raw)
            dest_zone   = normalize_zone(dest_raw)

            source_field = persistent_state["me_field"] if source_side == "me" else persistent_state["op_field"]
            dest_field   = persistent_state["me_field"] if dest_side == "me" else persistent_state["op_field"]

            # STRICT: must already exist in the source zone (and be a dict)
            card_object = source_field.get(source_zone)

            if card_object is None:
                print(f'❌ MOVE ERROR: source empty: {source_raw} -> {source_side}_field[{source_zone}] is None')
            elif not isinstance(card_object, dict):
                print(f'❌ MOVE ERROR: source not a card dict: {source_raw} -> {card_object!r}')
            elif card_object.get("name") != card_name:
                print(f'❌ MOVE ERROR: name mismatch at {source_raw}: found "{card_object.get("name")}", expected "{card_name}"')
            else:
                # remove from source
                source_field[source_zone] = None

                # place into destination (overwrite whatever is there; strict move)
                dest_field[dest_zone] = card_object

                print(f'✔ Moved "{card_name}" {source_raw} ({source_side}) → {dest_raw} ({dest_side})')

    elif play_type in ("SS ATK", "SS DEF"):
        print(f"✅ Processing play: {play}")

        log = (play["log"].get("private_log") or play["log"].get("public_log") or "").strip()

        # Examples:
        # Special Summoned "Terrorking Salmon" from hand (2/6) to M-3 (ATK)
        # Special Summoned "Terrorking Salmon" from GY to M-2 (DEF)
        # Special Summoned "Terrorking Salmon" from Deck to M-1 (ATK)
        pattern = r'^Special Summoned\s+"([^"]+)"\s+from\s+(.+?)\s+to\s+(M-\d+)\s+\((ATK|DEF)\)$'
        m = re.search(pattern, log)

        if not m:
            print(f"⚠️ Could not parse SS log: {log!r}")
            continue

        card_name = m.group(1)
        source_raw = m.group(2).strip()
        dest_zone  = m.group(3)
        pos        = m.group(4)

        # normalize source (handles "hand (2/6)")
        if source_raw.startswith("hand"):
            source = "hand"
        elif source_raw.startswith("Deck"):
            source = "Deck"
        elif source_raw.startswith("GY"):
            source = "GY"
        else:
            source = source_raw

        summoner_is_me = (play.get("username") == player_1)

        # summoner zones
        if summoner_is_me:
            s_hand = persistent_state["me_hand"]
            s_field = persistent_state["me_field"]
            s_gy = persistent_state["me_GY"]
            o_gy = persistent_state["op_GY"]
            owner_tag = "p1"
        else:
            s_hand = persistent_state["op_hand"]
            s_field = persistent_state["op_field"]
            s_gy = persistent_state["op_GY"]
            o_gy = persistent_state["me_GY"]
            owner_tag = "p2"

        target_oid = play.get("card", {}).get("object_id")

        card_object = None

        # ---------------- FROM HAND (must be summoner's hand) ----------------
        if source == "hand":
            # Prefer object_id match
            if target_oid is not None:
                for i, c in enumerate(s_hand):
                    if c.get("object_id") == target_oid:
                        card_object = s_hand.pop(i)
                        break
            # Fallback to name match within SAME hand only
            if card_object is None:
                for i, c in enumerate(s_hand):
                    if c.get("name") == card_name:
                        card_object = s_hand.pop(i)
                        break

            if card_object is None:
                print(f'❌ SS ERROR: could not find "{card_name}" (object_id={target_oid}) in HAND of summoner')
                continue

        # ---------------- FROM GY (can be summoner OR opponent GY) ----------------
        elif source == "GY":
            # 1) Try summoner GY first
            if target_oid is not None:
                for i, c in enumerate(s_gy):
                    if c.get("object_id") == target_oid:
                        card_object = s_gy.pop(i)
                        break
            if card_object is None:
                for i, c in enumerate(s_gy):
                    if c.get("name") == card_name:
                        card_object = s_gy.pop(i)
                        break

            # 2) If not found, try OTHER player's GY (steal / revive opponent's monster)
            if card_object is None:
                if target_oid is not None:
                    for i, c in enumerate(o_gy):
                        if c.get("object_id") == target_oid:
                            card_object = o_gy.pop(i)
                            break
                if card_object is None:
                    for i, c in enumerate(o_gy):
                        if c.get("name") == card_name:
                            card_object = o_gy.pop(i)
                            break

            if card_object is None:
                print(
                    f'❌ SS ERROR: could not find "{card_name}" (object_id={target_oid}) '
                    f'in either GY (summoner or opponent)'
                )
                continue

        # ---------------- FROM DECK (deck not enumerated, so create the entering card) ----------------
        elif source == "Deck":
            card_object = {
                "name": card_name,
                "object_id": target_oid,
                "battle_position": pos,
                "face_up": True,
                "owner": owner_tag,
            }

            if summoner_is_me:
                persistent_state["me_cards_in_deck"] = max(0, persistent_state["me_cards_in_deck"] - 1)
            else:
                persistent_state["op_cards_in_deck"] = max(0, persistent_state["op_cards_in_deck"] - 1)

        else:
            print(f"❌ SS ERROR: unknown source {source_raw!r} in log {log!r}")
            continue

        # ---------------- PLACE ON FIELD ----------------
        # IMPORTANT: owner stays whatever the card already had (do NOT overwrite on steals)
        card_object["battle_position"] = pos
        card_object["face_up"] = True
        card_object.setdefault("owner", owner_tag)

        s_field[dest_zone] = card_object
        print(f'✔ SS placed "{card_name}" to {dest_zone} ({pos}) by {play.get("username")}')
        
    else:
        print(f"❌ Processing play: {play}")
        # print(json.dumps(play, indent=2))
    ACTION = f"{play.get('username', '')}, {play['play']}"
    STATE_ACTION_PAIRS.append({
        "state": copy.deepcopy(persistent_state),
        "action": ACTION
    }) 


    

⏭️ Skipping play: {'play': 'RPS', 'player1_choice': 'Paper', 'seconds': 19, 'winner': 'seant856', 'player1': 'red-gadget', 'player2': 'seant856', 'log': {'public_log': 'Won Rock-Paper-Scissors', 'type': 'game', 'username': 'seant856'}, 'action': 'Duel', 'player2_choice': 'Scissors'}
✅ Processing play: {'play': 'Pick first', 'date': '2010-04-25', 'seconds': 28, 'cards': [{'def': '2500', 'monster_color': 'Effect', 'arrows': '', 'is_effect': 1, 'scale': 0, 'pic': '1', 'type': 'Fiend', 'ocg': 1, 'atk': '2700', 'tcg': 1, 'id': 1956, 'attribute': 'DARK', 'ability': '', 'pendulum': 0, 'flip': 0, 'level': 7, 'custom': 0, 'serial_number': '44330098', 'card_type': 'Monster', 'tcg_limit': 3, 'ocg_limit': 3, 'rush': 0, 'object_id': 20, 'effect': 'When you take damage from a card in your opponent\'s possession: You can Special Summon this card from your hand. You must control no cards to activate and to resolve this effect. When this card is Special Summoned by its effect: Activate the appropriate 

In [9]:
run_replay_viewer(STATE_ACTION_PAIRS)

KeyboardInterrupt: 